# 2 4 data summary

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()


In [ ]:
import pandas as pd

ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip')
source_features = pd.read_parquet('data/source_post_features.parquet')
source_features = source_features.merge(ts_posts[['tweet_id']].drop_duplicates(), on='tweet_id', how='inner')
source_treated = source_features.query('treated==1').reset_index(drop=True)
source_control = source_features.query('treated==0').reset_index(drop=True)
print(f'# Source posts&{{{source_features.shape[0]:,}}}&{{{source_treated.shape[0]:,}}}&{{{source_control.shape[0]:,}}}')
print(f'# Accounts&{{{len(source_features.user_id.unique()):,}}}&{{{len(source_treated.user_id.unique()):,}}}&{{{len(source_control.user_id.unique()):,}}}')
common_accounts = [account for account in source_treated.user_id.unique() if account in source_control.user_id.unique()]
print(len(common_accounts))

# reply_label is supplied as a processed indicator; source text is excluded.
print(source_features.query('reply_label==1').shape[0], source_features['reply_label'].mean().round(3))

In [ ]:
note_counts = source_treated.drop_duplicates(subset=['user_id'])[['user_id','total_corrections']].reset_index(drop=True)
print(f'# Note frequency&&{{{note_counts["total_corrections"].mean():.3f} ({note_counts["total_corrections"].std():.3f})}}&')

In [ ]:
print(f'# Days from roll-out&&{{{source_treated["days_from_rollout"].mean():.3f} ({source_treated["days_from_rollout"].std():.3f})}}&')

In [ ]:
followers = source_features.drop_duplicates(subset=['user_id'])['user_follower_count']
followers_treated = source_treated.drop_duplicates(subset=['user_id'])['user_follower_count']
followers_control = source_control.drop_duplicates(subset=['user_id'])['user_follower_count']
print(f'# Followers&{{{followers.mean():,.3f}}}&{{{followers_treated.mean():,.3f}}}&{{{followers_control.mean():,.3f}}}')
print(f'&{{({followers.std():,.3f})}}&{{({followers_treated.std():,.3f})}}&{{({followers_control.std():,.3f})}}')

In [ ]:
followees = source_features.drop_duplicates(subset=['user_id'])['user_following_count']
followees_treated = source_treated.drop_duplicates(subset=['user_id'])['user_following_count']
followees_control = source_control.drop_duplicates(subset=['user_id'])['user_following_count']
print(f'# Followees&{{{followees.mean():,.3f}}}&{{{followees_treated.mean():,.3f}}}&{{{followees_control.mean():,.3f}}}')
print(f'&{{({followees.std():,.3f})}}&{{({followees_treated.std():,.3f})}}&{{({followees_control.std():,.3f})}}')

In [ ]:
print(f'Account age&{{{source_features.user_account_age_days.mean():,.3f}}}&{{{source_treated.user_account_age_days.mean():,.3f}}}&{{{source_control.user_account_age_days.mean():,.3f}}}')
print(f'&{{({source_features.user_account_age_days.std():,.3f})}}&{{({source_treated.user_account_age_days.std():,.3f})}}&{{({source_control.user_account_age_days.std():,.3f})}}')

In [ ]:
vratio = source_features.drop_duplicates(subset=['user_id'])['user_is_verified'].mean()
vratio_treated = source_treated.drop_duplicates(subset=['user_id'])['user_is_verified'].mean()
vratio_control = source_control.drop_duplicates(subset=['user_id'])['user_is_verified'].mean()
print(f'Verified&{{{vratio:.1%}}}&{{{vratio_treated:.1%}}}&{{{vratio_control:.1%}}}')

In [ ]:
partisan = source_features.drop_duplicates(subset=['user_id'])['partisan_score']
partisan_treated = source_treated.drop_duplicates(subset=['user_id'])['partisan_score']
partisan_control = source_control.drop_duplicates(subset=['user_id'])['partisan_score']
print(f'Partisan score&{{{partisan.mean():,.3f}}}&{{{partisan_treated.mean():,.3f}}}&{{{partisan_control.mean():,.3f}}}')
print(f'&{{({partisan.std():,.3f})}}&{{({partisan_treated.std():,.3f})}}&{{({partisan_control.std():,.3f})}}')

In [ ]:
misexpo = source_features.drop_duplicates(subset=['user_id'])['misinfo_exposure_score']
misexpo_treated = source_treated.drop_duplicates(subset=['user_id'])['misinfo_exposure_score']
misexpo_control = source_control.drop_duplicates(subset=['user_id'])['misinfo_exposure_score']
print(f'Misinfo exposure score&{{{misexpo.mean():,.3f}}}&{{{misexpo_treated.mean():,.3f}}}&{{{misexpo_control.mean():,.3f}}}')
print(f'&{{({misexpo.std():,.3f})}}&{{({misexpo_treated.std():,.3f})}}&{{({misexpo_control.std():,.3f})}}')

In [ ]:
ts_posts_treated = ts_posts.merge(source_treated[['tweet_id']], on='tweet_id', how='inner').reset_index(drop=True)
ts_posts_control = ts_posts.merge(source_control[['tweet_id']], on='tweet_id', how='inner').reset_index(drop=True)
print(f'# Daily counts&{{{ts_posts.shape[0]:,}}}&{{{ts_posts_treated.shape[0]:,}}}&{{{ts_posts_control.shape[0]:,}}}')

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip').reset_index(drop=True)
ts_posts_treated = ts_posts.merge(source_treated[['tweet_id']], on='tweet_id', how='inner').reset_index(drop=True)
ts_posts_control = ts_posts.merge(source_control[['tweet_id']], on='tweet_id', how='inner').reset_index(drop=True)
print(f'# Weekly counts&{{{ts_posts.shape[0]:,}}}&{{{ts_posts_treated.shape[0]:,}}}&{{{ts_posts_control.shape[0]:,}}}')

In [ ]:
original_posts = pd.read_parquet('data/original_post_counts_by_account.parquet')
print('Retrieved original posts before account membership filtering:', original_posts['original_post_count'].sum())
original_posts.head()


In [ ]:
original_posts = original_posts.merge(source_features.drop_duplicates(subset=['user_id'])[['user_id']], on='user_id', how='inner')
original_posts_treated = original_posts.merge(source_treated.drop_duplicates(subset=['user_id'])[['user_id']], on='user_id', how='inner')
original_posts_control = original_posts.merge(source_control.drop_duplicates(subset=['user_id'])[['user_id']], on='user_id', how='inner')
print(f'# Original posts&{{{original_posts["original_post_count"].sum():,}}}&{{{original_posts_treated["original_post_count"].sum():,}}}&{{{original_posts_control["original_post_count"].sum():,}}}')

In [ ]:
for col in ['UserPostFrequency', 'UserPostMisleadingness', 'UserPostPositive', 'UserPostNegative', 'UserPostToxicity', 'UserPostConfidence']:
    print(f'{col}&{{{source_features.query("prior_corrections==0")[col].mean():,.3f}}}&{{{source_treated.query("prior_corrections==0")[col].mean():,.3f}}}&{{{source_control.query("prior_corrections==0")[col].mean():,.3f}}}')
    print(f'&{{({source_features.query("prior_corrections==0")[col].std():,.3f})}}&{{({source_treated.query("prior_corrections==0")[col].std():,.3f})}}&{{({source_control.query("prior_corrections==0")[col].std():,.3f})}}')

In [ ]:
pratio = source_features.query("prior_corrections==0")['UserPostPolitics'].mean()
pratio_treated = source_treated.query("prior_corrections==0")['UserPostPolitics'].mean()
pratio_control = source_control.query("prior_corrections==0")['UserPostPolitics'].mean()
print(f'UserPostPolitics&{{{pratio:.1%}}}&{{{pratio_treated:.1%}}}&{{{pratio_control:.1%}}}')

### Balance check

In [ ]:
from utils import check_balance, fit_columns

balance = check_balance(source_features, fit_columns, tab_export=True)

In [ ]:
balance.round(3)

### Note frequency

In [ ]:
print(f'Note==1: {note_counts.query("total_corrections==1").shape[0]:,}, {note_counts.query("total_corrections==1").shape[0]/note_counts.shape[0]:.1%}')
print(f'{source_treated.query("total_corrections==1").shape[0]:,}, {source_treated.query("total_corrections==1").shape[0]/source_treated.shape[0]:.1%}')

In [ ]:
for i in range(2, 6):
    print(f'Note>={i}: {note_counts.query(f"total_corrections>={i}").shape[0]:,}, {note_counts.query(f"total_corrections>={i}").shape[0]/note_counts.shape[0]:.1%}')
    print(f'{source_treated.query(f"total_corrections>={i}").shape[0]:,}, {source_treated.query(f"total_corrections>={i}").shape[0]/source_treated.shape[0]:.1%}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from utils import FONT_SIZE, LW, main_blue, sky_blue

plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.figure(layout="constrained")
sns.pointplot(x=np.arange(5), y=[0.641, 0.359, 0.217, 0.153, 0.119], linewidth=LW, linestyle=':', label='Accounts', color=sky_blue, marker='D')
sns.pointplot(x=np.arange(5), y=[0.238, 0.762, 0.677, 0.622, 0.587], linewidth=LW, linestyle='--', label='Posts', color=main_blue, marker='s')
plt.xticks(np.arange(5), [1,'$\\geq$2','$\\geq$3','$\\geq$4','$\\geq$5'])
plt.ylim(-0.08, 1.08)
plt.xlabel('Note frequency')
plt.ylabel('Ratio')
plt.savefig('outputs/figures/note_ratio.pdf')
plt.show()

In [ ]:
note_counts['total_corrections'].describe()

In [ ]:
source_treated['days2previousNote'].describe()

In [ ]:
plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.figure(layout="constrained")
sns.ecdfplot(note_counts, x='total_corrections', linewidth=LW, color=main_blue, complementary=True, log_scale=(False, True))
plt.ylim(0.00005, 1.8)
plt.xlabel('Note frequency')
plt.ylabel('CCDF')
plt.savefig('outputs/figures/note_ccdf.pdf')
plt.show()

In [ ]:
plt.style.use('default')
plt.rcParams['font.size'] = FONT_SIZE
plt.figure(layout="constrained")
sns.ecdfplot(source_treated, x='days2previousNote', linewidth=LW, color=main_blue, complementary=True, log_scale=(False, True))
# plt.xlim(-20, 660)
plt.ylim(0.00005, 1.8)
plt.xlabel('Days to previous note')
plt.ylabel('CCDF')
plt.savefig('outputs/figures/days2previousNote_ccdf.pdf')
plt.show()

### Statistics of prior corrections

In [ ]:
# Percentages use all focal events in each group
prior_counts = pd.crosstab(
    source_features['prior_corrections'].clip(upper=5), source_features['treated']
).reindex(index=range(6), columns=[0, 1], fill_value=0)
group_totals = source_features['treated'].value_counts()
prior_summary = pd.DataFrame(index=prior_counts.index)
for group, label in [(0, 'Control, n (%)'), (1, 'Treatment, n (%)')]:
    prior_summary[label] = prior_counts[group].map(lambda n: f'{n:,}') + ' (' + (
        prior_counts[group] / group_totals.get(group, float('nan')) * 100
    ).map(lambda pct: f'{pct:.1f}%') + ')'
prior_summary = prior_summary.rename(index={5: '≥5'})
prior_summary.index.name = '# Prior corrections'
display(prior_summary)

In [ ]:
print(prior_summary.to_latex())

### Fixed-horizon correction-frequency descriptives (60, 90 and 180 days)

In [ ]:
# Follow-up and horizon flags were calculated before pseudonymization.
# Coverage ends on June 11, 2024, inclusive (UTC).
fixed_horizon_accounts = pd.read_parquet('data/fixed_horizon_accounts.parquet')
horizons = (4, 28, 60, 90, 180)
rows = []
for days in horizons:
    eligible = fixed_horizon_accounts[f'eligible_{days}d']
    denominator = int(eligible.sum())
    repeated = int(fixed_horizon_accounts.loc[eligible, f'repeated_within_{days}d'].sum())
    rows.append({'horizon_days': days, 'sample_accounts': len(fixed_horizon_accounts),
        'missing_history': int(fixed_horizon_accounts['missing_history'].sum()),
        'insufficient_followup': int((~eligible & ~fixed_horizon_accounts['missing_history']).sum()),
        'eligible_accounts': denominator, 'repeated_accounts': repeated,
        'no_repeat_accounts': denominator-repeated,
        'repeated_percent_eligible': 100*repeated/denominator if denominator else float('nan')})
fixed_horizon_summary = pd.DataFrame(rows)
display(fixed_horizon_accounts['available_followup_days'].describe(percentiles=[.25,.5,.75]).to_frame('available_followup_days'))
display(fixed_horizon_summary)
followup_by_cohort = fixed_horizon_accounts.groupby('first_correction_quarter').agg(
    accounts=('user_id','size'), median_followup_days=('available_followup_days','median'),
    eligible_4d=('eligible_4d','sum'), eligible_28d=('eligible_28d','sum'),
    eligible_60d=('eligible_60d','sum'), eligible_90d=('eligible_90d','sum'), eligible_180d=('eligible_180d','sum'))
display(followup_by_cohort)
fixed_horizon_common_accounts = fixed_horizon_accounts.loc[fixed_horizon_accounts['eligible_180d']].copy()
common_rows = []
for horizon in horizons:
    n_accounts = len(fixed_horizon_common_accounts)
    n_repeated = int(fixed_horizon_common_accounts[f'repeated_within_{horizon}d'].sum())
    common_rows.append({'horizon_days':horizon,'eligible_accounts':n_accounts,
        'repeated_accounts':n_repeated,'no_repeat_accounts':n_accounts-n_repeated,
        'repeated_percent_eligible':100*n_repeated/n_accounts if n_accounts else float('nan')})
fixed_horizon_common_summary = pd.DataFrame(common_rows)
display(fixed_horizon_common_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5), layout='constrained')
ax.hist(fixed_horizon_accounts['available_followup_days'].dropna(), bins=25, edgecolor='white', color='grey')
ax.axvline(28, linestyle='-.', label='28 days', linewidth=LW, color='black')
ax.axvline(60, linestyle='-', label='60 days', linewidth=LW, color='black')
ax.axvline(90, linestyle='--', label='90 days', linewidth=LW, color='black')
ax.axvline(180, linestyle=':', label='180 days', linewidth=LW, color='black')
ax.set(xlabel='Available days after first observed correction', ylabel='# Accounts')
ax.legend()
plt.savefig('outputs/figures/followup_days_dist.pdf')
plt.show()

In [ ]:
from matplotlib.ticker import PercentFormatter, MultipleLocator

plot_all = fixed_horizon_summary.sort_values('horizon_days').set_index('horizon_days')
plot_common = fixed_horizon_common_summary.set_index('horizon_days').reindex(plot_all.index)
positions = np.arange(len(plot_all))
width = 0.34

fig, ax = plt.subplots(figsize=(8.5, 5), layout='constrained')
bars_all = ax.bar(positions,
                  plot_all['repeated_percent_eligible'], width=width,
                  color='grey', edgecolor='black', linewidth=1, hatch='//', label='Eligible at each horizon', zorder=3)
# bars_common = ax.bar(positions + width / 2,
#                      plot_common['repeated_percent_eligible'], width=width,
#                      color='grey', edgecolor='black', linewidth=1,
#                      hatch='//', label='Same accounts (180-day follow-up)', zorder=3)
# for bars in (bars_all, bars_common):
ax.bar_label(bars_all, labels=[f'{bar.get_height():.1f}%' for bar in bars_all],
             padding=5, fontsize=FONT_SIZE-3, color='black')
ax.set_xticks(positions)
ax.set_xticklabels([f'{h}' for h in plot_all.index])
ax.set_xlabel('Classification horizon (days)')
ax.set_ylabel('Repeated correction\namong eligible accounts')
ax.set_ylim(0, 105)
ax.yaxis.set_major_locator(MultipleLocator(20))
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.grid(axis='y', color='lightgrey', linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis='both', length=0, pad=8)
# ax.legend(fontsize=FONT_SIZE-4)
fig.savefig('outputs/figures/fixed_horizon_repeat_correction.pdf')
plt.show()

### Correction frequency over extended observation

In [ ]:
# Processed account totals from the original 60-day-eligible sample and
# the September 27, 2026 historical snapshot. No handles are released.
GROUPS = ['One correction', 'Repeated corrections']
accounts = pd.read_parquet('data/extended_correction_accounts.parquet')
print(f'{len(accounts):,} accounts in the extended-history comparison')
accounts.head()


### Correction-count distributions by original group

In [ ]:
count_columns = ['total_corrections', 'total_corrections_extended']
assert accounts['user_id'].is_unique, 'Expected one row per account.'

correction_distribution = accounts.melt(
    id_vars=['user_id', 'original_group'], value_vars=count_columns,
    var_name='history', value_name='corrections'
)
display(correction_distribution.groupby(['original_group', 'history'])['corrections']
        .describe(percentiles=[.25, .5, .75, .9]).round(3))
display(accounts.groupby('original_group')[count_columns].agg(lambda x: x.isna().sum())
        .rename(columns=lambda x: f'{x}: missing'))

In [ ]:
from utils import single_color, repeated_color

fig, ax = plt.subplots(figsize=(8.5, 5), layout="constrained")
sns.ecdfplot(data=accounts, x='total_corrections', hue='original_group', linewidth=LW, hue_order=GROUPS, palette=[single_color, repeated_color], complementary=True, log_scale=(True, False), ax=ax)
ax.get_legend().set_title('60-day follow-up')
ax.set(ylim=(-0.08, 1.08), xlabel='Total corrections (original)', ylabel='CCDF')
fig.savefig('outputs/figures/correction_ccdf_original.pdf')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5), layout="constrained")
sns.ecdfplot(data=accounts, x='total_corrections_extended', hue='original_group', linewidth=LW, hue_order=GROUPS, palette=[single_color, repeated_color], complementary=True, log_scale=(True, False), ax=ax)
ax.get_legend().set_title('60-day follow-up')
ax.set(ylim=(-0.08, 1.08), xlabel='Total corrections (extended)', ylabel='CCDF')
fig.savefig('outputs/figures/correction_ccdf_extended.pdf')
plt.show()

In [ ]:
from scipy.stats import ks_2samp
def get_ksresult(d1, d2):
    ksresult = ks_2samp(d1, d2)
    p_text = '< 0.001' if ksresult.pvalue < 0.001 else f'= {ksresult.pvalue:.3f}'
    print(f'KS = {ksresult.statistic:.3f}, p {p_text}')
get_ksresult(accounts['total_corrections'][accounts['original_group']=='One correction'],
             accounts['total_corrections'][accounts['original_group']=='Repeated corrections'])

In [ ]:
get_ksresult(accounts['total_corrections_extended'][accounts['original_group']=='One correction'],
             accounts['total_corrections_extended'][accounts['original_group']=='Repeated corrections'])